----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Content layer - Tradefeedr - Parent Orders
This standalone notebook demonstrates how to retrieve existing FX parent-order analytics using `lseg.data.content.tradefeedr.ParentOrderDefinition`, select fields and request data asynchronously.

The tests use `ld.tradefeedr.get_fx_algo_parent_orders`, an Access layer wrapper around this Content definition. The wrapper returns a DataFrame directly; the Content layer returns a response with `data.df` and `data.raw`.

**Prerequisites:** a running LSEG Workspace desktop session configured for the Data Library and access to Tradefeedr parent-order data. The tests use a desktop session. The historical period and initial field selection below match those tests; adapt the period to the data available to your account. An empty response may mean that no orders are available in that period. Service and entitlement errors are not suppressed.

**Confidentiality:** parent-order results may include trader, account, subaccount and execution details. Only request data you are authorized to access, and clear outputs before sharing the notebook. These examples retrieve analytics; they do not submit, modify or cancel orders.

See [Pre-Trade Forecast](EX-2.13.01-Tradefeedr-PreTradeForecast.ipynb) for forecast scenarios.

#### Learn more

Visit the [LSEG Developer Portal](https://developers.lseg.com/en) for learning materials about the LSEG Data Library for Python.

#### Getting Help and Support

For questions about the API, visit the [LSEG Developer Community](https://community.developers.lseg.com/).

----

## Set the configuration file location
Run the notebook from its own directory. The project's `Configuration` directory contains `lseg-data.config.json`; keep session settings and the application key there rather than in the notebook.

In [ ]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../../Configuration"

## Some Imports to start with

In [ ]:
import lseg.data as ld
from lseg.data.content import tradefeedr

## Open the data session
`open_session()` selects the configured default session. Set that default to your desktop session to reproduce the test environment.

In [ ]:
ld.open_session()

## Retrieve parent orders with selected fields
`start` and `end` define the service's `within` filter on `Date`. The request builder converts them to `YYYY-MM-DD` dates; use date ranges rather than relying on time-of-day precision.

The field list below is the one exercised by the active test. Omitting `fields` uses the library's much broader default selection, including quantities, prices, execution scores and performance measures. Explicitly selecting fields makes the requested data easier to review.

In [ ]:
start = "2020-01-01"
end = "2024-01-01"
fields = [
    "Date",
    "ParentOrderID",
    "Symbol",
    "TradeCurrency",
    "Side",
    "LegSymbols",
    "Trader",
    "Account",
    "SubAccount",
    "AlgoName",
    "LP",
    "TimeZoneArrive",
    "HourArrive",
    "TimeZoneEnd",
]

response = tradefeedr.ParentOrderDefinition(
    start=start,
    end=end,
    fields=fields,
).get_data()
response.data.df

### Inspect the returned DataFrame
The DataFrame contains the selected fields. Show its dimensions and column names without copying all order details into another output. The original service payload is also available through `response.data.raw`; it is not displayed here because it can contain sensitive order data.

In [ ]:
orders = response.data.df
print(f"Rows: {len(orders)}, columns: {len(orders.columns)}")
print(orders.columns.tolist())
if orders.empty:
    print("No parent orders returned; check the date range and account data availability.")

## Request execution analytics
The following fields come from the library's default parent-order selection. This is an additional example based on the implementation, rather than a separate test scenario. It retrieves USD order and trade quantities, duration, execution score and arrival-mid performance for the same period. Units are part of some field names, for example `USD` and `BPS`; consult the Tradefeedr service documentation for their precise definitions.

In [ ]:
analytics_fields = [
    "Date", "ParentOrderID", "Symbol", "Side",
    "OrderQuantityUSD", "TradeQuantityUSD", "ArrivalTime",
    "Duration", "ExecutionScore", "ArrivalMidPerfBPS",
]
analytics_response = tradefeedr.ParentOrderDefinition(
    start=start,
    end=end,
    fields=analytics_fields,
).get_data()
analytics_response.data.df

## Retrieve parent orders asynchronously
The Content layer also exposes `get_data_async()`. This example extends the synchronous test with an asynchronous request for the same date range and a smaller field selection. Jupyter supports top-level `await`; request errors propagate to the cell.

In [ ]:
async_response = await tradefeedr.ParentOrderDefinition(
    start=start,
    end=end,
    fields=["Date", "ParentOrderID", "Symbol", "AlgoName"],
).get_data_async()
async_response.data.df

### Advanced request options
`ParentOrderDefinition` also accepts `extended_params` for backend options. The active test passes `None`, so these examples deliberately use the supported date and field parameters without inventing additional filters.

In the local implementation, extended options may be supplied with or without an outer `options` object. A `within` filter on `Date` replaces the date filter built from `start` and `end`; other filters are appended. Use only backend-documented options and avoid unintentionally overriding the date range or selected fields.

## Close the session
No orders were changed and no files were created. If an earlier request fails, run this cell to close the session. Clear all order-data outputs before sharing the notebook.

In [ ]:
ld.close_session()